# Tabular Pipelines: Classification and Regression

Build leakage-safe ColumnTransformer pipelines, tune nested parameters, and evaluate held-out classification and regression.

- **Study time:** 50-65 minutes
- **Prerequisites:** pandas, train/test splitting, and basic supervised metrics
- **Mode:** `quick`
- **Data policy:** no external files or downloads; a mixed-type synthetic table is created in memory
- **Provenance:** consolidated from the curated sklearn pipeline notebook and three legacy pipeline code printouts

Output convention: every retained textual result begins with a label that identifies the operation that produced it.


In [1]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, f1_score, mean_squared_error, r2_score
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

rng = np.random.default_rng(41)


def show(label, value):
    print(f"\n--- {label} ---\n{value}")

## 1. Build one mixed-type feature table and two targets


In [2]:
n_rows = 700
frame = pd.DataFrame(
    {
        "age": rng.integers(20, 70, size=n_rows).astype(float),
        "income": rng.normal(70_000, 20_000, size=n_rows),
        "rooms": rng.integers(1, 7, size=n_rows).astype(float),
        "region": rng.choice(["north", "south", "east", "west"], size=n_rows),
        "risk": rng.choice(["low", "medium", "high"], size=n_rows, p=[0.45, 0.35, 0.20]),
    }
)
frame.loc[rng.choice(n_rows, 45, replace=False), "income"] = np.nan
frame.loc[rng.choice(n_rows, 25, replace=False), "region"] = None

risk_score = frame["risk"].map({"low": 0.0, "medium": 0.8, "high": 1.6})
region_score = (
    frame["region"].map({"north": 0.3, "south": -0.1, "east": 0.15, "west": 0.0}).fillna(0)
)
income_filled = frame["income"].fillna(frame["income"].median())
purchase_logit = (
    -4.0 + 0.000045 * income_filled + 0.025 * frame["age"] - 0.9 * risk_score + region_score
)
purchase_probability = 1.0 / (1.0 + np.exp(-purchase_logit))
y_class = (rng.random(n_rows) < purchase_probability).astype(int)
y_reg = (
    30_000
    + 90 * income_filled
    + 8_000 * frame["rooms"]
    - 12_000 * risk_score
    + 20_000 * region_score
    + rng.normal(0, 8_000, size=n_rows)
)

show("Dataset | feature shape", frame.shape)
show("Dataset | dtypes", frame.dtypes.to_string())
show("Dataset | missing values", frame.isna().sum().to_string())
show(
    "Classification target | class fractions",
    pd.Series(y_class).value_counts(normalize=True).sort_index().to_string(),
)


--- Dataset | feature shape ---
(700, 5)

--- Dataset | dtypes ---
age       float64
income    float64
rooms     float64
region        str
risk          str

--- Dataset | missing values ---
age        0
income    45
rooms      0
region    25
risk       0

--- Classification target | class fractions ---
0    0.517143
1    0.482857


## 2. Define preprocessing once


In [3]:
numeric_features = ["age", "income", "rooms"]
nominal_features = ["region"]
ordinal_features = ["risk"]

numeric_pipeline = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)
nominal_pipeline = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)
ordinal_pipeline = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "ordinal",
            OrdinalEncoder(
                categories=[["low", "medium", "high"]],
                handle_unknown="use_encoded_value",
                unknown_value=-1,
            ),
        ),
    ]
)

preprocessor = ColumnTransformer(
    [
        ("numeric", numeric_pipeline, numeric_features),
        ("nominal", nominal_pipeline, nominal_features),
        ("ordinal", ordinal_pipeline, ordinal_features),
    ]
)
show(
    "Preprocessing | feature groups",
    {"numeric": numeric_features, "nominal": nominal_features, "ordinal": ordinal_features},
)


--- Preprocessing | feature groups ---
{'numeric': ['age', 'income', 'rooms'], 'nominal': ['region'], 'ordinal': ['risk']}


## 3. Classification: baseline, pipeline, and nested search

A dummy model verifies that learned signal beats a trivial policy. This synthetic table is IID, so a stratified random split is appropriate; use grouped or chronological splits when rows share entities or time dependence.


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    frame,
    y_class,
    test_size=0.25,
    random_state=42,
    stratify=y_class,
)
classification_pipeline = Pipeline(
    [
        ("preprocess", preprocessor),
        ("model", RandomForestClassifier(random_state=42, n_jobs=1)),
    ]
)
classification_search = RandomizedSearchCV(
    classification_pipeline,
    param_distributions={
        "model__n_estimators": [40, 80, 120],
        "model__max_depth": [None, 5, 10],
        "model__min_samples_leaf": [1, 3, 6],
    },
    n_iter=4,
    scoring="f1",
    cv=3,
    random_state=42,
    n_jobs=1,
)
classification_search.fit(X_train, y_train)
class_prediction = classification_search.predict(X_test)
classification_baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
baseline_class_prediction = classification_baseline.predict(X_test)
baseline_f1 = f1_score(y_test, baseline_class_prediction)
classification_f1 = f1_score(y_test, class_prediction)

show("Classification | train/test shapes", (X_train.shape, X_test.shape))
show("Classification | best parameters", classification_search.best_params_)
show(
    "Classification | baseline versus tuned held-out F1",
    {"dummy_most_frequent": baseline_f1, "random_forest": classification_f1},
)
show(
    "Classification | held-out report",
    classification_report(y_test, class_prediction, digits=3, zero_division=0),
)


--- Classification | train/test shapes ---
((525, 5), (175, 5))

--- Classification | best parameters ---
{'model__n_estimators': 80, 'model__min_samples_leaf': 3, 'model__max_depth': 5}

--- Classification | baseline versus tuned held-out F1 ---
{'dummy_most_frequent': 0.0, 'random_forest': 0.6589595375722543}

--- Classification | held-out report ---
              precision    recall  f1-score   support

           0      0.686     0.648     0.667        91
           1      0.640     0.679     0.659        84

    accuracy                          0.663       175
   macro avg      0.663     0.663     0.663       175
weighted avg      0.664     0.663     0.663       175



## 4. Regression: baseline and pipeline using the same preprocessing contract


In [5]:
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    frame,
    y_reg,
    test_size=0.25,
    random_state=42,
)
regression_pipeline = Pipeline(
    [
        ("preprocess", preprocessor),
        ("model", RandomForestRegressor(random_state=42, n_jobs=1)),
    ]
)
regression_search = RandomizedSearchCV(
    regression_pipeline,
    param_distributions={
        "model__n_estimators": [40, 80, 120],
        "model__max_depth": [None, 6, 12],
        "model__min_samples_leaf": [1, 3, 6],
    },
    n_iter=4,
    scoring="neg_mean_squared_error",
    cv=3,
    random_state=42,
    n_jobs=1,
)
regression_search.fit(X_train_reg, y_train_reg)
regression_prediction = regression_search.predict(X_test_reg)
rmse = np.sqrt(mean_squared_error(y_test_reg, regression_prediction))
regression_baseline = DummyRegressor(strategy="mean").fit(X_train_reg, y_train_reg)
baseline_regression_prediction = regression_baseline.predict(X_test_reg)
baseline_rmse = np.sqrt(mean_squared_error(y_test_reg, baseline_regression_prediction))

show("Regression | best parameters", regression_search.best_params_)
show(
    "Regression | baseline versus tuned held-out RMSE",
    {"dummy_mean": baseline_rmse, "random_forest": rmse},
)
show("Regression | held-out R2", r2_score(y_test_reg, regression_prediction))


--- Regression | best parameters ---
{'model__n_estimators': 40, 'model__min_samples_leaf': 1, 'model__max_depth': 6}

--- Regression | baseline versus tuned held-out RMSE ---
{'dummy_mean': np.float64(1754682.6248458198), 'random_forest': np.float64(61550.06871642431)}

--- Regression | held-out R2 ---
0.9987656897590418


## 5. Inspect the fitted feature space


In [6]:
fitted_preprocessor = classification_search.best_estimator_.named_steps["preprocess"]
feature_names = fitted_preprocessor.get_feature_names_out()
transformed_sample = fitted_preprocessor.transform(X_test.head(3))

show("Pipeline inspection | transformed feature names", feature_names)
show("Pipeline inspection | transformed sample shape", transformed_sample.shape)
show("Pipeline inspection | nested parameter prefix example", "model__max_depth")


--- Pipeline inspection | transformed feature names ---
['numeric__age' 'numeric__income' 'numeric__rooms' 'nominal__region_east'
 'nominal__region_north' 'nominal__region_south' 'nominal__region_west'
 'ordinal__risk']

--- Pipeline inspection | transformed sample shape ---
(3, 8)

--- Pipeline inspection | nested parameter prefix example ---
model__max_depth


## 6. Retrieval checks


In [7]:
assert len(class_prediction) == len(X_test)
assert len(regression_prediction) == len(X_test_reg)
assert transformed_sample.shape[0] == 3
assert np.isfinite(rmse)
assert classification_f1 > baseline_f1
assert rmse < baseline_rmse

show("Pipeline checks | status", "all shape and held-out-evaluation assertions passed")


--- Pipeline checks | status ---
all shape and held-out-evaluation assertions passed
